# Kaggle v20 SFT Reproduction Notebook

この notebook は **A-Open-ProgressPrizePublication/README.md** と
**A-Open-ProgressPrizePublication/nemotron/training/sft/04-08-16-14/** に残っている
v20 勝ちランのスナップショットを使って、Kaggle GPU 上で **Tinker を使わずに**
学習を再現するための実装です。

## 方針

- **使う学習データ**: `training/sft/04-08-16-14/tokens/` と `logprobs/index.jsonl`
- **使う設定**: v20 の `config.json` に合わせる
- **学習開始点**: 既存 adapter への継ぎ足しではなく、ベースモデルに対する新規 LoRA SFT
- **使わないもの**: Tinker API、`notebook_tinker.py` の SVD 変換
- **Kaggle 向けに置き換えるもの**: Tinker の学習ループだけをローカル PyTorch + PEFT に置き換える

## 再現で厳密に合わせるもの

- 7,830 examples
- 27,850,703 tokens
- batch size 32 の step 構成
- 1 epoch
- cross-entropy
- learning rate 2e-4 の線形減衰
- LoRA rank 32
- max length 8192
- train_attn / train_mlp / train_unembed に対応する target modules

## Kaggle ローカル実装として調整するもの

- `LOCAL_MICRO_BATCH_SIZE` は単一 GPU メモリに合わせて調整する
- optimizer step は v20 の 32-example batch をそのまま維持する
- 保存される adapter は最初から提出互換の target modules を使うので SVD 変換は不要


In [ ]:
import glob
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def module_exists(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def run(cmd: list[str]) -> None:
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

def kaggle_input_root() -> Path:
    root = Path("/kaggle/input")
    return root if root.exists() else Path(".")

def find_local_wheel(pattern: str) -> str | None:
    matches = sorted(kaggle_input_root().rglob(pattern))
    return str(matches[0]) if matches else None

GENERIC_WHEEL_DIRS = [
    "/kaggle/input/datasets/mayukh18/nemotron-packages/packages",
    "/kaggle/input/datasets/llkh0a/rtx-wheels/wheels",
]

missing_generic = [
    pkg for pkg in ["transformers", "peft", "safetensors"]
    if not module_exists(pkg)
]
if missing_generic:
    wheel_dirs = [p for p in GENERIC_WHEEL_DIRS if Path(p).exists()]
    if not wheel_dirs:
        raise RuntimeError(
            f"Missing packages {missing_generic}, and no local wheel dir was found under /kaggle/input"
        )
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index"]
    for wheel_dir in wheel_dirs:
        cmd.extend(["--find-links", wheel_dir])
    cmd.extend(missing_generic)
    run(cmd)

for pattern in ["causal_conv1d-*.whl", "mamba_ssm-*.whl"]:
    package_name = pattern.split("-")[0]
    if not module_exists(package_name):
        wheel = find_local_wheel(pattern)
        if wheel is None:
            raise RuntimeError(f"Missing required wheel for {package_name}: {pattern}")
        run([sys.executable, "-m", "pip", "install", "-q", wheel])

print("Environment setup complete.")


In [ ]:
import json
import math
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime
from functools import lru_cache

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

V20_RUN_NAME = "04-08-16-14"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
BASE_LR = 2e-4
BETAS = (0.9, 0.95)
EPS = 1e-8
WEIGHT_DECAY = 0.0
MAX_LENGTH = 8192
NUM_EPOCHS = 1
GLOBAL_BATCH_SIZE = 32
LOCAL_MICRO_BATCH_SIZE = int(os.environ.get("LOCAL_MICRO_BATCH_SIZE", "1"))
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
SAVE_RAW_LOGPROBS = os.environ.get("SAVE_RAW_LOGPROBS", "1") == "1"
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "in_proj",
    "out_proj",
    "up_proj",
    "down_proj",
    "lm_head",
]

EXPECTED_EXAMPLES = 7830
EXPECTED_TOTAL_TOKENS = 27850703
EXPECTED_MASKED_TOKENS = 1281896
EXPECTED_UNMASKED_TOKENS = 26568807
EXPECTED_TOTAL_STEPS = 245

def discover_snapshot_root() -> Path:
    candidates: list[Path] = []
    local_dev = Path(
        "/Users/mac-studio/work/NVIDIA Nemotron Model Reasoning Challenge/"
        "A-Open-ProgressPrizePublication/nemotron/training/sft/04-08-16-14"
    )
    if local_dev.exists():
        candidates.append(local_dev)
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for config_path in kaggle_root.glob(f"**/training/sft/{V20_RUN_NAME}/config.json"):
            candidates.append(config_path.parent)
    if not candidates:
        raise FileNotFoundError(
            f"Could not find training/sft/{V20_RUN_NAME}/config.json under /kaggle/input"
        )
    return candidates[0]

def discover_model_path() -> str:
    candidates = [
        Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"),
        Path("/kaggle/input/nemotron-3-nano-30b-a3b-bf16"),
        Path("/kaggle/input/nvidia-nemotron-3-nano-30b-a3b-bf16"),
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    return BASE_MODEL_NAME

SNAPSHOT_ROOT = discover_snapshot_root()
NEMOTRON_ROOT = SNAPSHOT_ROOT.parents[3]
MODEL_PATH = discover_model_path()
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
RUN_ROOT = WORK_ROOT / f"kaggle-v20-sft-repro-{datetime.now().strftime('%m%d-%H%M%S')}"
ADAPTER_DIR = RUN_ROOT / "adapter"
LOGPROB_ROOT = RUN_ROOT / "logprobs"
RAW_LOGPROB_EPOCH_DIR = LOGPROB_ROOT / "0"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RAW_LOGPROB_EPOCH_DIR.mkdir(parents=True, exist_ok=True)

print("SNAPSHOT_ROOT:", SNAPSHOT_ROOT)
print("NEMOTRON_ROOT:", NEMOTRON_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("RUN_ROOT:", RUN_ROOT)
print("LOCAL_MICRO_BATCH_SIZE:", LOCAL_MICRO_BATCH_SIZE)


In [ ]:
@dataclass(frozen=True)
class ExampleRef:
    problem_id: str
    segment: str
    category: str
    step: int
    num_loss_tokens: int
    token_path: str

def read_jsonl(path: Path):
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def load_v20_batches(snapshot_root: Path):
    step_to_examples: dict[int, list[ExampleRef]] = defaultdict(list)
    category_counter: Counter[str] = Counter()
    for row in read_jsonl(snapshot_root / "logprobs" / "index.jsonl"):
        token_path = snapshot_root / "tokens" / row["problem_id"] / f"{Path(row['segment']).stem}.json"
        if not token_path.exists():
            raise FileNotFoundError(f"Missing token snapshot: {token_path}")
        ex = ExampleRef(
            problem_id=row["problem_id"],
            segment=row["segment"],
            category=row["category"],
            step=int(row["step"]),
            num_loss_tokens=int(row["num_loss_tokens"]),
            token_path=str(token_path),
        )
        step_to_examples[ex.step].append(ex)
        category_counter[ex.category] += 1
    step_batches = [step_to_examples[i] for i in sorted(step_to_examples)]
    return step_batches, category_counter

@lru_cache(maxsize=128)
def load_token_snapshot(token_path: str):
    payload = json.loads(Path(token_path).read_text())
    return payload["tokens"], payload["mask"]

def validate_v20_snapshot(step_batches: list[list[ExampleRef]]):
    total_examples = sum(len(batch) for batch in step_batches)
    if total_examples != EXPECTED_EXAMPLES:
        raise AssertionError(f"Expected {EXPECTED_EXAMPLES} examples, found {total_examples}")
    if len(step_batches) != EXPECTED_TOTAL_STEPS:
        raise AssertionError(f"Expected {EXPECTED_TOTAL_STEPS} steps, found {len(step_batches)}")
    total_tokens = 0
    total_masked = 0
    total_unmasked = 0
    max_len = 0
    for batch in step_batches:
        for ex in batch:
            tokens, mask = load_token_snapshot(ex.token_path)
            total_tokens += len(tokens)
            total_masked += sum(1 for x in mask if x == 0)
            total_unmasked += sum(1 for x in mask if x == 1)
            max_len = max(max_len, len(tokens))
            if sum(mask) != ex.num_loss_tokens:
                raise AssertionError(
                    f"num_loss_tokens mismatch for {ex.problem_id}: {sum(mask)} != {ex.num_loss_tokens}"
                )
    if total_tokens != EXPECTED_TOTAL_TOKENS:
        raise AssertionError(f"Expected {EXPECTED_TOTAL_TOKENS} total tokens, found {total_tokens}")
    if total_masked != EXPECTED_MASKED_TOKENS:
        raise AssertionError(f"Expected {EXPECTED_MASKED_TOKENS} masked tokens, found {total_masked}")
    if total_unmasked != EXPECTED_UNMASKED_TOKENS:
        raise AssertionError(f"Expected {EXPECTED_UNMASKED_TOKENS} unmasked tokens, found {total_unmasked}")
    if max_len > MAX_LENGTH:
        raise AssertionError(f"Found sequence longer than {MAX_LENGTH}: {max_len}")
    return {
        "num_examples": total_examples,
        "total_tokens": total_tokens,
        "total_masked_tokens": total_masked,
        "total_unmasked_tokens": total_unmasked,
        "total_steps": len(step_batches),
        "max_seq_len": max_len,
    }

def chunked(items, chunk_size):
    for i in range(0, len(items), chunk_size):
        yield items[i:i + chunk_size]

def collate_microbatch(examples: list[ExampleRef], pad_token_id: int):
    token_rows = []
    label_rows = []
    lengths = []
    for ex in examples:
        tokens, mask = load_token_snapshot(ex.token_path)
        tokens = tokens[:MAX_LENGTH]
        mask = mask[:MAX_LENGTH]
        labels = [tok if m == 1 else -100 for tok, m in zip(tokens, mask)]
        token_rows.append(tokens)
        label_rows.append(labels)
        lengths.append(len(tokens))
    max_len = max(lengths)
    batch_size = len(examples)
    input_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)
    for i, (tokens, label_ids) in enumerate(zip(token_rows, label_rows)):
        seq_len = len(tokens)
        input_ids[i, :seq_len] = torch.tensor(tokens, dtype=torch.long)
        attention_mask[i, :seq_len] = 1
        labels[i, :seq_len] = torch.tensor(label_ids, dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "lengths": lengths,
        "examples": examples,
    }

def save_raw_logprobs(epoch_dir: Path, ex: ExampleRef, logprobs: list[float]) -> None:
    path = epoch_dir / ex.problem_id / f"{Path(ex.segment).stem}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        f.write(json.dumps({"logprobs": [round(v, 4) for v in logprobs]}) + "\n")

def set_optimizer_lr(optimizer, lr: float) -> None:
    for group in optimizer.param_groups:
        group["lr"] = lr

STEP_BATCHES, CATEGORY_COUNTS = load_v20_batches(SNAPSHOT_ROOT)
SNAPSHOT_STATS = validate_v20_snapshot(STEP_BATCHES)
print("Snapshot stats:", SNAPSHOT_STATS)
print("Category counts:")
for category, count in sorted(CATEGORY_COUNTS.items()):
    print(f"  {category:26s} {count}")
print("First 3 batch sizes:", [len(b) for b in STEP_BATCHES[:3]])
print("Last batch size:", len(STEP_BATCHES[-1]))


In [ ]:
assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."
device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise ValueError("Tokenizer must expose eos_token_id or pad_token_id")
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

target_hits = {name: 0 for name in TARGET_MODULES}
for module_name, _ in model.named_modules():
    for target_name in TARGET_MODULES:
        if module_name.endswith(target_name):
            target_hits[target_name] += 1
print("Matched target modules:")
for name, hits in target_hits.items():
    print(f"  {name:12s} {hits}")
missing = [name for name, hits in target_hits.items() if hits == 0]
if missing:
    raise AssertionError(f"Missing target module hits: {missing}")

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, peft_config)
model.to(device)
model.print_trainable_parameters()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=BASE_LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY,
)

run_config = {
    "base_model_name": BASE_MODEL_NAME,
    "resolved_model_path": MODEL_PATH,
    "snapshot_root": str(SNAPSHOT_ROOT),
    "v20_snapshot": True,
    "training_start": "base-model-direct-lora-sft",
    "warm_start_adapter": None,
    "num_epochs": NUM_EPOCHS,
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "local_micro_batch_size": LOCAL_MICRO_BATCH_SIZE,
    "learning_rate": BASE_LR,
    "betas": BETAS,
    "eps": EPS,
    "weight_decay": WEIGHT_DECAY,
    "max_length": MAX_LENGTH,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "save_raw_logprobs": SAVE_RAW_LOGPROBS,
    "stats": SNAPSHOT_STATS,
    "category_counts": dict(sorted(CATEGORY_COUNTS.items())),
    "source_v20_config": json.loads((SNAPSHOT_ROOT / "config.json").read_text()),
}
with (RUN_ROOT / "config.json").open("w") as f:
    json.dump(run_config, f, indent=2)

metrics_path = RUN_ROOT / "metrics.jsonl"
loss_path = RUN_ROOT / "loss.jsonl"
index_path = LOGPROB_ROOT / "index.jsonl"
checkpoints_path = RUN_ROOT / "checkpoints.jsonl"

epoch_loss_sum = 0.0
epoch_token_count = 0
global_step = 0

model.train()
with metrics_path.open("w") as metrics_file, loss_path.open("w") as loss_file, index_path.open("w") as index_file:
    for epoch in range(NUM_EPOCHS):
        for step_idx, step_examples in enumerate(STEP_BATCHES):
            step_start = time.time()
            lr = BASE_LR * max(0.0, 1.0 - (global_step / EXPECTED_TOTAL_STEPS))
            set_optimizer_lr(optimizer, lr)
            optimizer.zero_grad(set_to_none=True)

            step_loss_sum = 0.0
            step_token_count = 0
            cat_loss: dict[str, float] = defaultdict(float)
            cat_tokens: dict[str, int] = defaultdict(int)
            cat_min_logprob: dict[str, float] = {}
            step_total_active_tokens = sum(ex.num_loss_tokens for ex in step_examples)
            if step_total_active_tokens <= 0:
                continue

            for micro_examples in chunked(step_examples, LOCAL_MICRO_BATCH_SIZE):
                batch = collate_microbatch(micro_examples, tokenizer.pad_token_id)
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                )
                logits = outputs.logits[:, :-1, :].float()
                target_tokens = input_ids[:, 1:]
                target_labels = labels[:, 1:]
                valid_mask = target_labels.ne(-100)

                log_probs = torch.log_softmax(logits, dim=-1)
                token_logprobs = torch.gather(log_probs, -1, target_tokens.unsqueeze(-1)).squeeze(-1)
                nll_sum = (-(token_logprobs) * valid_mask).sum()
                (nll_sum / step_total_active_tokens).backward()

                step_loss_sum += float(nll_sum.detach().cpu())
                step_token_count += int(valid_mask.sum().item())

                token_logprobs_cpu = token_logprobs.detach().cpu()
                valid_mask_cpu = valid_mask.detach().cpu()
                for row_idx, ex in enumerate(batch["examples"]):
                    target_len = batch["lengths"][row_idx] - 1
                    ex_logprobs = token_logprobs_cpu[row_idx, :target_len].tolist()
                    ex_valid = valid_mask_cpu[row_idx, :target_len].tolist()
                    if SAVE_RAW_LOGPROBS:
                        save_raw_logprobs(RAW_LOGPROB_EPOCH_DIR, ex, ex_logprobs)
                    unmasked_lps = [lp for lp, keep in zip(ex_logprobs, ex_valid) if keep]
                    total_loss = float(sum(-lp for lp in unmasked_lps))
                    min_logprob = float(min(unmasked_lps)) if unmasked_lps else 0.0
                    cat_loss[ex.category] += total_loss
                    cat_tokens[ex.category] += len(unmasked_lps)
                    if ex.category not in cat_min_logprob or min_logprob < cat_min_logprob[ex.category]:
                        cat_min_logprob[ex.category] = min_logprob
                    index_record = {
                        "epoch": epoch,
                        "step": step_idx,
                        "problem_id": ex.problem_id,
                        "segment": ex.segment,
                        "category": ex.category,
                        "num_loss_tokens": len(unmasked_lps),
                        "total_loss": round(total_loss, 4),
                        "min_logprob": round(min_logprob, 4),
                    }
                    index_file.write(json.dumps(index_record) + "\n")

                del outputs, logits, target_tokens, target_labels, valid_mask, log_probs, token_logprobs, nll_sum
                torch.cuda.empty_cache()

            optimizer.step()

            elapsed = time.time() - step_start
            metric_record = {
                "epoch": epoch,
                "step": step_idx,
                "lr": lr,
                "n": len(step_examples),
                "elapsed": round(elapsed, 2),
                "time": datetime.now().strftime("%m-%d-%H-%M"),
                "_loss_per_token": (step_loss_sum / step_token_count) if step_token_count else None,
            }
            for category in sorted(cat_loss):
                if cat_tokens[category] > 0:
                    metric_record[f"_loss_per_token/{category}"] = cat_loss[category] / cat_tokens[category]
                    metric_record[f"_min_logprob/{category}"] = round(cat_min_logprob[category], 4)
            metrics_file.write(json.dumps(metric_record) + "\n")
            metrics_file.flush()
            index_file.flush()

            epoch_loss_sum += step_loss_sum
            epoch_token_count += step_token_count
            global_step += 1
            print(
                f"epoch={epoch} step={step_idx:03d}/{EXPECTED_TOTAL_STEPS} "
                f"lr={lr:.6f} n={len(step_examples)} "
                f"loss_per_token={metric_record['_loss_per_token']:.6f} "
                f"elapsed={elapsed:.1f}s"
            )

    avg_nll = epoch_loss_sum / epoch_token_count
    loss_record = {
        "epoch": 0,
        "metrics": [
            [{"nll_per_token": round(avg_nll, 6)}],
            [{"perplexity": round(math.exp(min(avg_nll, 20.0)), 4)}],
        ],
    }
    loss_file.write(json.dumps(loss_record) + "\n")

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)

adapter_config_path = ADAPTER_DIR / "adapter_config.json"
adapter_config = json.loads(adapter_config_path.read_text())
adapter_config["base_model_name_or_path"] = BASE_MODEL_NAME
adapter_config["target_modules"] = TARGET_MODULES
adapter_config_path.write_text(json.dumps(adapter_config, indent=2))

submission_zip = RUN_ROOT / "submission.zip"
with zipfile.ZipFile(submission_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(ADAPTER_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(ADAPTER_DIR))

with checkpoints_path.open("w") as f:
    f.write(json.dumps({
        "name": "final",
        "adapter_dir": str(ADAPTER_DIR),
        "submission_zip": str(submission_zip),
        "run_root": str(RUN_ROOT),
    }) + "\n")

summary = {
    "run_root": str(RUN_ROOT),
    "adapter_dir": str(ADAPTER_DIR),
    "submission_zip": str(submission_zip),
    "num_examples": SNAPSHOT_STATS["num_examples"],
    "total_steps": SNAPSHOT_STATS["total_steps"],
    "total_tokens": SNAPSHOT_STATS["total_tokens"],
    "local_micro_batch_size": LOCAL_MICRO_BATCH_SIZE,
    "save_raw_logprobs": SAVE_RAW_LOGPROBS,
}
with (RUN_ROOT / "run_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)

print("Done.")
print(json.dumps(summary, indent=2))
